# Using Project Source Code

This chapter demonstrates how to import library code from `src/` without
`sys.path` hacks. The project is installed in editable mode when you run
`uv sync`, so notebooks can import packages directly.

The example pipeline:

1. Generate synthetic data with a small simulation
2. Train a random forest
3. Compute **confidence intervals** via case bootstrap (re-fit on resampled training data)
4. Compute **prediction intervals** via MAPIE split conformal prediction
5. Visualize everything with Altair

## Imports

No path bootstrapping is required — the packages are installed by `uv sync`.

In [1]:
from sklearn.model_selection import train_test_split

from analysis import bootstrap_confidence_intervals
from core import Settings, TrainingData, build_split_dataset
from prediction import (
    conformal_intervals,
    fit_conformal,
    fit_random_forest,
    predict,
    random_forest_regressor,
)
from simulation import generate_dataset
from visualization import plot_dataset, plot_intervals

d:\repositories\jupyter-book-template\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate synthetic data

In [2]:
settings = Settings(n_samples=1500, seed=0, noise_heteroscedasticity=20, n_estimators=200, max_depth=5)
data = generate_dataset(settings)
data.head()

,x,y
0,2.739234,6.373780
1,-4.604266,-12.387189
2,-9.180530,-13.231512
3,-9.669447,-11.510018
4,6.265405,-2.772410


In [3]:
plot_dataset(data)

alt.LayerChart(...)

## Split into train, calibration, and test sets

In [4]:
train, remainder = train_test_split(data, test_size=0.3, random_state=0)
calib, test = train_test_split(remainder, test_size=0.5, random_state=0)
split_data = build_split_dataset(
    TrainingData.validate(train),
    TrainingData.validate(calib),
    TrainingData.validate(test),
)
regressor = random_forest_regressor(settings)
len(train), len(calib), len(test)

(1050, 225, 225)

## Fit a random forest

Before fitting, `x` is expanded into polynomial powers (up to `settings.polynomial_degree`)
and Fourier harmonics (up to `settings.fourier_terms`, using `settings.seasonality_frequency`).
The random forest then sees this richer feature matrix instead of raw `x` alone.

In [5]:
model = fit_random_forest(split_data, regressor, settings)
predictions = predict(model, split_data, settings)

## Bootstrap confidence intervals (case bootstrap)

Re-fit a random forest on each bootstrap sample of the training data, then take
per-`x` quantiles of the resulting predictions. This estimates uncertainty in
**E[Y | X]** (the fitted mean function), not individual observation noise.

In [6]:
ci = bootstrap_confidence_intervals(regressor, split_data, settings)
ci.head()

Bootstrap resamples: 100%|██████████| 200/200 [00:19<00:00, 10.39it/s]


,x,lower,upper,kind
0,-3.935980,-9.889872,-7.450168,confidence
1,-1.354701,7.289750,8.940694,confidence
2,3.924302,1.059202,4.626997,confidence
3,-7.003689,-18.522639,-17.829586,confidence
4,5.243433,-1.451523,0.517216,confidence


## Split conformal prediction intervals (MAPIE)

In [7]:
conformal_model = fit_conformal(split_data, regressor, settings)
pi = conformal_intervals(conformal_model, split_data, settings)
pi.head()

,x,lower,upper,kind
0,-3.935980,-14.089837,-3.421672,prediction
1,-1.354701,2.593205,13.261370,prediction
2,3.924302,-2.709128,7.959037,prediction
3,-7.003689,-23.657962,-12.989798,prediction
4,5.243433,-6.004059,4.664105,prediction


## Visualize with Altair

In [8]:
plot_intervals(split_data, predictions, ci, pi)

alt.LayerChart(...)

## Tests

Each module under `src/` has a matching test module under `tests/`.
Run the full suite with:

```bash
uv run poe test
```

CI runs tests before building the book (`uv run poe ci`).